###### Boilerplate

In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

os.chdir(PROJECT_DIR)

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

print("DumLLM initialized!")
print("Working directory:", os.getcwd())
print("Project files:", os.listdir(PROJECT_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DumLLM initialized!
Working directory: /content/drive/MyDrive/DumLLM
Project files: ['requirements.txt', '.venv', '.gitignore', '.git', 'notebooks', 'src', 'start_DumLLM.ipynb']


# 2. Working with Text data

## 2.2 Tokenizing Text

In [ ]:
from pathlib import Path

file_path = Path("public/the_verdict.txt")
raw_text = file_path.read_text(encoding="utf-8")

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [ ]:
len(preprocessed)

4690

## 2.3 Converting tokens into token IDs

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"Vocab size: {vocab_size}")

Vocab size: 1130


In [ ]:
vocab = {word:i for i,word in enumerate(all_words)}
print(vocab)

{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance': 82, 'Ri

In [ ]:
import re

class SimpleTokenizerV1:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s,i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    ids = [self.str_to_int[token] for token in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[id] for id in ids])
    # replace spaces before the specified punctuations
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

In [ ]:
text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [ ]:
original_text = tokenizer.decode(ids)
original_text

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Adding special context tokens

In [ ]:
text = "Hello, do you like tea. is this-- a test?"
tokenizer.encode(text)

KeyError: 'Hello'

In [ ]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>","<|unk|>"])

vocab = {token:i for i,token in enumerate(all_tokens)}
print(vocab)

{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance': 82, 'Ri

In [ ]:
class SimpleTokenizerV2:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s,i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [
        item.strip() for item in preprocessed if item.strip()
    ]
    # adding rule, if word not available replace with <|unk|>
    preprocessed = [
        item if item in self.str_to_int
        else "<|unk|>" for item in preprocessed
    ]
    ids = [self.str_to_int[token] for token in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[id] for id in ids])
    # replace spaces before the specified punctuations
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
    return text

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

In [ ]:
text = "Hello, do you like tea. is this-- a test?"
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 7, 584, 999, 6, 115, 1131, 10]

In [ ]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea. is this -- a <|unk|>?'

## 2.5 Byte pair encoding



In [ ]:
import tiktoken

In [ ]:
tiktoken.__version__

'0.14.0'

In [ ]:
tokenizer = tiktoken.encoding_for_model("gpt2")

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
tokenizer.encode(text,allowed_special={"<|endoftext|>"})

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 1659,
 617,
 34680,
 27271,
 13]

## 2.6 Data sampling with a sliding window

In [ ]:
with open(file_path, "r", encoding="utf-8") as f:
  raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [ ]:
enc_sample = enc_text[50:]

In [ ]:
context_size = 4 # gpt2 had 1024
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [ ]:
for i in range(1,context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(tokenizer.decode(context), " --> ", tokenizer.decode([desired]))

 and  -->   established
 and established  -->   himself
 and established himself  -->   in
 and established himself in  -->   a


In [ ]:
import torch
torch.__version__

'2.11.0+cpu'

In [ ]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []           # if very large dataset, out of memory
    self.target_ids = []

    # tokenizer entire text
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    # use sliding window to chunk the text into overlapping sequence of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
  # initializer tokenizer
  tokenizer = tiktoken.encoding_for_model("gpt2")

  # create dataset
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

  # create dataloader
  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      drop_last=drop_last,
      num_workers=num_workers,
  )

  return dataloader

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("INPUTS: \n", inputs)
print("TARGETS: \n", targets)

INPUTS: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
TARGETS: 
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.7 Creating token embeddings

In [ ]:
inputs_ids = torch.tensor([ 2,   3,    5,   1])

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [ ]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035],
        [-0.5880,  0.3486,  0.6603],
        [-0.2196, -0.3792,  0.7671],
        ...,
        [-0.5931,  1.0895, -0.6854],
        [ 0.7447,  0.5803, -0.4246],
        [-0.3130,  0.7558, -1.2656]], requires_grad=True)


In [ ]:
embedding_layer(inputs_ids)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

## 2.8 Encoding word positions

In [ ]:
vocab_size = tokenizer.n_vocab
output_dim = 256 # gpt2 had 1024

token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print("Token IDs:\n",inputs)
print("Inputs shape:\n",inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Inputs shape:
 torch.Size([8, 4])


In [ ]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

In [ ]:
token_embeddings[0, 0]

tensor([ 1.7660, -1.4702, -1.5674,  0.9020, -0.2525,  0.4977,  0.3826,  0.8998,
        -0.9327,  0.2263, -1.4592,  0.5888, -1.9706,  0.2106, -0.0921,  0.3441,
        -0.2705, -0.0593,  0.0117,  0.1635, -1.8002, -0.5781,  0.7211,  1.2537,
         1.0199,  1.6373,  0.7089, -0.4291,  1.1104,  2.2836,  0.6346,  0.0936,
         0.0354,  0.4868, -0.0771,  1.5380,  0.4837, -0.7077, -0.5757, -1.8185,
        -0.9909,  1.2571,  0.6615, -1.3672,  0.4810,  1.1207, -0.4170, -2.0653,
        -0.0634,  1.2445,  0.1194, -0.5284, -0.6994,  1.3655,  0.9489,  0.2877,
         1.5369, -1.0373, -0.8084, -1.4553, -0.7534,  0.5711,  3.1662, -0.2721,
        -0.4500,  0.7485, -0.4019,  0.0598,  1.3011,  1.1048,  0.0692,  0.0996,
         1.4984,  0.1774, -0.7941, -0.2813, -0.1545, -1.3183,  0.6886,  0.8908,
         0.7273,  0.8676, -0.5354,  0.1384,  0.4806,  2.5442, -1.3060, -0.2178,
        -1.0573,  0.9542,  0.4276, -0.0499,  0.1188,  1.1291,  0.0290, -0.0696,
         0.5545, -0.1362, -0.3537,  0.76

In [ ]:
# creating position encoding
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
pos_embeddings.shape

torch.Size([4, 256])

In [ ]:
# adding position embeddings to the token embeddings (to all batchs)
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)
print(input_embeddings[0,0])

torch.Size([8, 4, 256])
tensor([-0.4199, -2.0724, -1.0424,  0.8015, -0.2175, -1.3087,  0.1834,  0.6434,
        -1.0859,  1.8445, -2.3814, -0.4996, -2.7686,  0.1842,  1.0450,  1.5301,
         1.3418, -0.1226,  0.1001,  0.2360, -0.6274,  0.0708,  0.5531, -0.4362,
         2.8938,  1.5687,  0.4248,  1.5342,  1.5997,  0.8993,  1.5659, -0.2812,
        -1.0757,  0.9299, -0.5216,  1.7142,  0.0261, -1.5298,  0.1220, -1.3028,
        -1.1426,  2.2649,  0.3390,  1.3531,  0.1866,  0.8964, -0.4799, -3.5564,
         0.1395,  2.1380,  0.7350,  1.0597,  0.3544,  1.4031,  1.7842,  2.0023,
         1.2440, -1.1829, -0.3337, -0.2555,  0.2296,  1.2409,  1.3362,  0.2979,
        -0.5560,  1.7777,  0.2943, -0.0195, -0.1920,  0.6169, -0.5918, -1.8195,
         1.7768,  1.9504, -0.0598, -0.7273,  0.4044, -0.9558,  1.6918,  1.3160,
         0.5954,  1.4007,  0.3392,  1.0722,  0.0694,  0.2933, -1.8501, -0.7793,
        -0.5234,  1.3769, -1.7588,  0.0724, -0.9476, -0.7498,  0.7467,  0.2097,
         2.0647,